<a href="https://colab.research.google.com/github/MerkulovDaniil/optim/blob/master/assets/Notebooks/Global_optimization_minimization_methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Глобальная оптимизация: интерактивные иллюстрации для задачи минимизации

В этом ноутбуке все алгоритмы решают задачу

$$
\min_{x\in\mathcal X} f(x),
$$

и имеют доступ только к значениям функции. Ноутбук сделан в стиле Colab: параметры в ячейках задаются через `#@param`, поэтому можно менять число точек, температуру, размер популяции, seed и перезапускать ячейки.

Список методов:

* Grid Search
* Random Search
* Simulated Annealing
* Hill Climbing
* Particle Swarm Optimization
* Ant Colony Optimization
* CMA-ES
* TPE
* MAP-Elites


In [ ]:
#@title Общие импорты, стиль и одномерная функция для минимизации
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import Ellipse
from IPython.display import HTML, display

plt.style.use("default")
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.labelsize": 13,
    "legend.fontsize": 10,
})

# sigma задаёт шум oracle: мы наблюдаем J(x)=f(x)+noise.
sigma = 0.0 #@param {type:"number"}


def raw_score(x):
    """Исходная функция из старого ноутбука: там была максимизация."""
    x = np.asarray(x)
    return np.where(
        (x - np.floor(x)) * np.abs(x + 5) < 0.2 * (np.abs(x) < 5),
        0.5 * np.floor(x),
        0.4 * np.exp(-2 * (x - 6)**2)
        + 2 * np.exp(-5 * (x + 4)**2)
        + 3 * np.exp(-0.5 * (x - 3)**4)
        + 0.5 * np.exp(-(x + 7)**2)
        + 0.1 * np.cos(x * 30)
        + 0.01 * np.abs(x - 7) * np.sin(x**3 / 13)
        - np.tanh(x**2)
        - 0.1 * np.abs(x),
    )


def f(x):
    """Цель минимизации: меняем знак у старой функции максимизации."""
    return -raw_score(x)


def J(x):
    x = np.asarray(x)
    return f(x) + sigma * np.random.normal(size=x.shape)


X = np.linspace(-10, 10, 1600)
Y = f(X)
Y_MIN, Y_MAX = float(Y.min()), float(Y.max())


def draw_1d(title="Целевая функция", size=(13, 5)):
    fig, ax = plt.subplots(figsize=size)
    ax.plot(X, Y, color="#1f2937", lw=2.0, label="целевая функция")
    ax.set_xlim(-10, 10)
    ax.set_ylim(Y_MIN - 0.35, Y_MAX + 0.45)
    ax.set_xlabel("$x$")
    ax.set_ylabel("$f(x)$")
    ax.set_title(title)
    ax.grid(linestyle=":", alpha=0.65)
    return fig, ax


def best_min(xs, values=None):
    xs = np.asarray(xs)
    ys = J(xs) if values is None else np.asarray(values)
    idx = int(np.argmin(ys))
    return xs[idx], ys[idx], idx


def show_video(anim):
    display(HTML(anim.to_html5_video()))
    plt.close(anim._fig)


fig, ax = draw_1d("Многомодальная функция для минимизации")
idx = int(np.argmin(Y))
ax.scatter([X[idx]], [Y[idx]], marker="*", s=180, color="#16a34a", edgecolor="black", label="лучший минимум на плотной сетке")
ax.legend(loc="upper right")
plt.show()


## Grid Search

Перебираем равномерную сетку и выбираем лучший узел. Метод прост и воспроизводим, но число вычислений растёт как произведение числа узлов по координатам.

In [ ]:
#@title Grid Search: статическая картинка
N_GRID = 41 #@param {type:"slider", min:5, max:401, step:2}

xs = np.linspace(-10, 10, N_GRID)
ys = J(xs)
bx, by, _ = best_min(xs, ys)

fig, ax = draw_1d(f"Grid Search: {N_GRID} узлов")
ax.scatter(xs, ys, marker="x", s=55, color="#ef4444", label="узлы сетки")
ax.scatter([bx], [by], marker="*", s=190, color="#16a34a", edgecolor="black", label="лучший узел")
ax.legend(loc="upper right")
plt.show()


In [ ]:
#@title Grid Search: анимация последовательного просмотра узлов
N_GRID_ANIM = 41 #@param {type:"slider", min:11, max:101, step:2}

xs = np.linspace(-10, 10, N_GRID_ANIM)
ys = J(xs)
fig, ax = draw_1d("Grid Search: узлы просматриваются по порядку")
seen = ax.scatter([], [], marker="x", s=55, color="#ef4444", label="проверено")
best = ax.scatter([], [], marker="*", s=190, color="#16a34a", edgecolor="black", label="лучшее")
ax.legend(loc="upper right")


def update(frame):
    cur_x = xs[: frame + 1]
    cur_y = ys[: frame + 1]
    seen.set_offsets(np.c_[cur_x, cur_y])
    bx, by, _ = best_min(cur_x, cur_y)
    best.set_offsets([[bx, by]])
    ax.set_title(f"Grid Search: проверено {frame + 1} из {N_GRID_ANIM} узлов")
    return seen, best

anim = animation.FuncAnimation(fig, update, frames=len(xs), interval=180, blit=False)
show_video(anim)


## Random Search

Сэмплируем точки независимо из заданного распределения. В отличие от сетки, случайный поиск не тратит весь бюджет на равномерное покрытие неважных координат.

In [ ]:
#@title Random Search: статическая картинка
N_RANDOM = 60 #@param {type:"slider", min:5, max:500, step:5}
SEED_RANDOM = 7 #@param {type:"integer"}

rng = np.random.default_rng(SEED_RANDOM)
xs = rng.uniform(-10, 10, N_RANDOM)
ys = J(xs)
bx, by, _ = best_min(xs, ys)

fig, ax = draw_1d(f"Random Search: {N_RANDOM} случайных проб")
ax.scatter(xs, ys, marker="x", s=55, color="#ef4444", label="случайные точки")
ax.scatter([bx], [by], marker="*", s=190, color="#16a34a", edgecolor="black", label="лучшее")
ax.legend(loc="upper right")
plt.show()


In [ ]:
#@title Random Search: анимация накопления случайных проб
N_RANDOM_ANIM = 70 #@param {type:"slider", min:10, max:200, step:5}
SEED_RANDOM_ANIM = 8 #@param {type:"integer"}

rng = np.random.default_rng(SEED_RANDOM_ANIM)
xs = rng.uniform(-10, 10, N_RANDOM_ANIM)
ys = J(xs)
fig, ax = draw_1d("Random Search: лучшая точка обновляется скачками")
seen = ax.scatter([], [], marker="x", s=55, color="#ef4444", label="пробы")
best = ax.scatter([], [], marker="*", s=190, color="#16a34a", edgecolor="black", label="лучшее")
ax.legend(loc="upper right")


def update(frame):
    cur_x = xs[: frame + 1]
    cur_y = ys[: frame + 1]
    seen.set_offsets(np.c_[cur_x, cur_y])
    bx, by, _ = best_min(cur_x, cur_y)
    best.set_offsets([[bx, by]])
    ax.set_title(f"Random Search: {frame + 1} проб")
    return seen, best

anim = animation.FuncAnimation(fig, update, frames=len(xs), interval=160, blit=False)
show_video(anim)


## Simulated Annealing

Локальный поиск, который иногда принимает ухудшения. Для минимизации переход $x	o x'$ принимается с вероятностью

$$
p=\min\left(1, \exp\left(-rac{f(x')-f(x)}{T}ight)ight).
$$

Высокая температура даёт exploration, низкая температура превращает метод в почти жадный локальный поиск.

In [ ]:
#@title Simulated Annealing: анимация
N_ITER_SA = 50 #@param {type:"slider", min:5, max:200, step:5}
T0 = 1.6 #@param {type:"number"}
COOLING = 0.93 #@param {type:"number"}
STEP_SA = 0.9 #@param {type:"number"}
SEED_SA = 9 #@param {type:"integer"}

rng = np.random.default_rng(SEED_SA)
x = float(rng.uniform(-9, 9))
states = []
for k in range(N_ITER_SA):
    T = max(1e-4, T0 * (COOLING ** k))
    cand = float(np.clip(x + rng.normal(scale=STEP_SA), -10, 10))
    delta = float(J([cand])[0] - J([x])[0])
    accepted = delta <= 0 or rng.random() < np.exp(-delta / T)
    states.append((x, cand, accepted, T))
    if accepted:
        x = cand

fig, ax = draw_1d("Simulated Annealing")
accepted_line, = ax.plot([], [], color="#16a34a", lw=1.8, marker="o", ms=4, label="принятые")
rejected_scatter = ax.scatter([], [], marker="s", s=55, facecolor="none", edgecolor="#ef4444", label="отклонённые")
current_scatter = ax.scatter([], [], marker="*", s=190, color="#16a34a", edgecolor="black", label="текущая")
ax.legend(loc="upper right")
accepted_x = [states[0][0]]
rejected_x = []


def update(frame):
    _, cand, ok, T = states[frame]
    if ok:
        accepted_x.append(cand)
    else:
        rejected_x.append(cand)
    accepted_line.set_data(accepted_x, J(accepted_x))
    if rejected_x:
        rejected_scatter.set_offsets(np.c_[rejected_x, J(rejected_x)])
    current_scatter.set_offsets([[accepted_x[-1], J([accepted_x[-1]])[0]]])
    ax.set_title(f"Simulated Annealing: T={T:.3f}, шаг {frame + 1}")
    return accepted_line, rejected_scatter, current_scatter

anim = animation.FuncAnimation(fig, update, frames=len(states), interval=220, blit=False)
show_video(anim)


## Hill Climbing

Жадный локальный поиск: генерируем несколько соседей и переходим только если нашли улучшение. Сам по себе метод локальный, но полезен как exploitation-блок и как часть random-restart схем.

In [ ]:
#@title Hill Climbing: анимация
N_ITER_HC = 35 #@param {type:"slider", min:5, max:150, step:5}
N_CANDIDATES_HC = 10 #@param {type:"slider", min:1, max:50, step:1}
STEP_HC = 0.8 #@param {type:"number"}
SEED_HC = 11 #@param {type:"integer"}

rng = np.random.default_rng(SEED_HC)
x = float(rng.uniform(-9, 9))
states = []
for k in range(N_ITER_HC):
    cand = np.clip(x + rng.normal(scale=STEP_HC, size=N_CANDIDATES_HC), -10, 10)
    cand_y = J(cand)
    best_idx = int(np.argmin(cand_y))
    x_new = float(cand[best_idx]) if cand_y[best_idx] < J([x])[0] else x
    states.append((x, cand, x_new))
    x = x_new

fig, ax = draw_1d("Hill Climbing")
candidates_scatter = ax.scatter([], [], marker="x", s=50, color="#ef4444", label="соседи")
path_line, = ax.plot([], [], color="#16a34a", marker="o", lw=2, ms=4, label="траектория")
ax.legend(loc="upper right")
path = [states[0][0]]


def update(frame):
    _, cand, x_new = states[frame]
    path.append(x_new)
    candidates_scatter.set_offsets(np.c_[cand, J(cand)])
    path_line.set_data(path, J(path))
    ax.set_title(f"Hill Climbing: шаг {frame + 1}")
    return candidates_scatter, path_line

anim = animation.FuncAnimation(fig, update, frames=len(states), interval=240, blit=False)
show_video(anim)


## Общие функции для двумерных методов

Для PSO и CMA-ES используем двумерные тестовые функции, чтобы было видно движение популяции на контурной карте.

In [ ]:
#@title Двумерные функции и отрисовка контуров

def rastrigin(z):
    z = np.asarray(z)
    return 20 + np.sum(z**2 - 10 * np.cos(2 * np.pi * z), axis=-1)


def himmelblau(z):
    z = np.asarray(z)
    x = z[..., 0]
    y = z[..., 1]
    return (x**2 + y - 11)**2 + (x + y**2 - 7)**2


def draw_rastrigin(ax, title="Rastrigin"):
    xs = np.linspace(-5.12, 5.12, 220)
    ys = np.linspace(-5.12, 5.12, 220)
    Xg, Yg = np.meshgrid(xs, ys)
    Z = rastrigin(np.dstack([Xg, Yg]))
    ax.contourf(Xg, Yg, Z, levels=35, cmap="Blues_r")
    ax.contour(Xg, Yg, Z, levels=12, colors="white", linewidths=0.45, alpha=0.75)
    ax.set_xlim(-5.12, 5.12)
    ax.set_ylim(-5.12, 5.12)
    ax.set_aspect("equal")
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.set_title(title)
    ax.grid(linestyle=":", alpha=0.35)


def draw_himmelblau(ax, title="Himmelblau"):
    xs = np.linspace(-5.5, 5.5, 240)
    ys = np.linspace(-5.5, 5.5, 240)
    Xg, Yg = np.meshgrid(xs, ys)
    Z = himmelblau(np.dstack([Xg, Yg]))
    ax.contourf(Xg, Yg, Z, levels=np.linspace(0, 260, 35), cmap="Blues_r", extend="max")
    ax.contour(Xg, Yg, Z, levels=[1, 5, 15, 35, 70, 120, 180], colors="white", linewidths=0.55, alpha=0.8)
    ax.set_xlim(-5.5, 5.5)
    ax.set_ylim(-5.5, 5.5)
    ax.set_aspect("equal")
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.set_title(title)
    ax.grid(linestyle=":", alpha=0.35)


## Particle Swarm Optimization

Частицы хранят скорость, личную лучшую точку и глобальную лучшую точку роя:

$$
v_i^{k+1}=\omega v_i^k+c_1r_1(p_i^k-x_i^k)+c_2r_2(g^k-x_i^k),\qquad
x_i^{k+1}=x_i^k+v_i^{k+1}.
$$

In [ ]:
#@title Particle Swarm Optimization: анимация на функции Rastrigin
N_PARTICLES = 24 #@param {type:"slider", min:5, max:80, step:1}
N_ITER_PSO = 55 #@param {type:"slider", min:5, max:150, step:5}
OMEGA = 0.62 #@param {type:"number"}
C1 = 1.25 #@param {type:"number"}
C2 = 1.35 #@param {type:"number"}
SEED_PSO = 13 #@param {type:"integer"}

rng = np.random.default_rng(SEED_PSO)
x = rng.uniform(-5, 5, size=(N_PARTICLES, 2))
v = rng.normal(scale=0.6, size=(N_PARTICLES, 2))
pbest = x.copy()
pbest_val = rastrigin(pbest)
states = []
for k in range(N_ITER_PSO):
    gbest = pbest[int(np.argmin(pbest_val))].copy()
    states.append((x.copy(), gbest.copy()))
    r1 = rng.random((N_PARTICLES, 2))
    r2 = rng.random((N_PARTICLES, 2))
    v = OMEGA * v + C1 * r1 * (pbest - x) + C2 * r2 * (gbest - x)
    x = np.clip(x + v, -5.12, 5.12)
    vals = rastrigin(x)
    improved = vals < pbest_val
    pbest[improved] = x[improved]
    pbest_val[improved] = vals[improved]
states.append((x.copy(), pbest[int(np.argmin(pbest_val))].copy()))

fig, ax = plt.subplots(figsize=(8, 7))
draw_rastrigin(ax, "Particle Swarm Optimization")
particles = ax.scatter([], [], s=36, color="#16a34a", label="частицы")
best = ax.scatter([], [], marker="*", s=220, color="#facc15", edgecolor="black", label="лучшее")
ax.legend(loc="upper right")


def update(frame):
    pts, g = states[frame]
    particles.set_offsets(pts)
    best.set_offsets([g])
    ax.set_title(f"Particle Swarm Optimization: поколение {frame}")
    return particles, best

anim = animation.FuncAnimation(fig, update, frames=len(states), interval=180, blit=False)
show_video(anim)


## Ant Colony Optimization

Муравьиный алгоритм особенно естественен для задач на графах. В примере ниже решается маленькая TSP-задача: феромон усиливается на коротких маршрутах и испаряется на каждом шаге.

In [ ]:
#@title Ant Colony Optimization: анимация для TSP
N_CITIES = 12 #@param {type:"slider", min:6, max:30, step:1}
N_ANTS = 28 #@param {type:"slider", min:5, max:100, step:1}
N_ITER_ACO = 55 #@param {type:"slider", min:5, max:150, step:5}
ALPHA_ACO = 1.3 #@param {type:"number"}
BETA_ACO = 3.0 #@param {type:"number"}
RHO_ACO = 0.18 #@param {type:"number"}
SEED_ACO = 17 #@param {type:"integer"}

rng = np.random.default_rng(SEED_ACO)
angles = np.linspace(0, 2 * np.pi, N_CITIES, endpoint=False)
points = np.c_[np.cos(angles), np.sin(angles)] * 4 + rng.normal(scale=0.45, size=(N_CITIES, 2))
dist = np.linalg.norm(points[:, None, :] - points[None, :, :], axis=-1) + np.eye(N_CITIES)
eta = 1 / dist
tau = np.ones_like(dist)
np.fill_diagonal(tau, 0)


def tour_length(tour):
    return sum(np.linalg.norm(points[tour[i]] - points[tour[(i + 1) % N_CITIES]]) for i in range(N_CITIES))

best_tour, best_len = None, np.inf
states = []
for it in range(N_ITER_ACO):
    tours = []
    for _ in range(N_ANTS):
        start = int(rng.integers(N_CITIES))
        tour = [start]
        while len(tour) < N_CITIES:
            i = tour[-1]
            allowed = np.array([j for j in range(N_CITIES) if j not in tour])
            weights = (tau[i, allowed] ** ALPHA_ACO) * (eta[i, allowed] ** BETA_ACO)
            probs = weights / weights.sum()
            tour.append(int(rng.choice(allowed, p=probs)))
        L = tour_length(tour)
        tours.append((tour, L))
        if L < best_len:
            best_len = L
            best_tour = tour[:]
    tau *= (1 - RHO_ACO)
    for tour, L in sorted(tours, key=lambda item: item[1])[: max(1, N_ANTS // 5)]:
        inc = 1 / L
        for i in range(N_CITIES):
            a, b = tour[i], tour[(i + 1) % N_CITIES]
            tau[a, b] += inc
            tau[b, a] += inc
    states.append((tau.copy(), best_tour[:], best_len))


def draw_aco_state(ax, tau_state, tour, best_len, iteration):
    ax.clear()
    ax.set_title(f"Ant Colony Optimization: итерация {iteration}, длина {best_len:.2f}")
    ax.set_aspect("equal")
    ax.axis("off")
    tmax = max(tau_state.max(), 1e-12)
    for i in range(N_CITIES):
        for j in range(i + 1, N_CITIES):
            lw = 0.25 + 4.0 * tau_state[i, j] / tmax
            ax.plot([points[i, 0], points[j, 0]], [points[i, 1], points[j, 1]], color="#9ca3af", lw=lw, alpha=0.28)
    cycle = tour + [tour[0]]
    ax.plot(points[cycle, 0], points[cycle, 1], color="#16a34a", lw=3.0, label="лучший маршрут")
    ax.scatter(points[:, 0], points[:, 1], s=95, color="white", edgecolor="#111827", linewidth=1.2, zorder=3)
    for k, p in enumerate(points):
        ax.text(p[0], p[1], str(k + 1), ha="center", va="center", fontsize=9, zorder=4)
    ax.legend(loc="upper right")

fig, ax = plt.subplots(figsize=(8, 7))

def update(frame):
    tau_state, tour, best_len = states[frame]
    draw_aco_state(ax, tau_state, tour, best_len, frame + 1)
    return []

anim = animation.FuncAnimation(fig, update, frames=len(states), interval=200, blit=False)
show_video(anim)


## CMA-ES

Упрощённая визуализация идеи CMA-ES: поддерживаем нормальное распределение кандидатов, выбираем элиту и адаптируем центр, масштаб и ковариацию. Это не полная промышленная реализация, но геометрия метода видна.

In [ ]:
#@title CMA-ES: анимация на функции Химмельблау
N_ITER_CMA = 45 #@param {type:"slider", min:5, max:120, step:5}
LAMBDA_CMA = 34 #@param {type:"slider", min:8, max:100, step:2}
ELITE_FRAC_CMA = 0.33 #@param {type:"number"}
SEED_CMA = 19 #@param {type:"integer"}

rng = np.random.default_rng(SEED_CMA)
mean = np.array([-4.4, 4.3], dtype=float)
sigma_cma = 1.25
cov = np.eye(2)
states = []
for k in range(N_ITER_CMA):
    samples = rng.multivariate_normal(mean, sigma_cma**2 * cov, size=LAMBDA_CMA)
    vals = himmelblau(samples)
    mu = max(2, int(ELITE_FRAC_CMA * LAMBDA_CMA))
    elite = samples[np.argsort(vals)[:mu]]
    old_mean = mean.copy()
    mean = 0.35 * mean + 0.65 * elite.mean(axis=0)
    centered = (elite - old_mean) / max(sigma_cma, 1e-8)
    empirical_cov = centered.T @ centered / len(elite) + 1e-4 * np.eye(2)
    cov = 0.75 * cov + 0.25 * empirical_cov
    if len(elite) > 2:
        sigma_cma = float(np.clip(0.86 * sigma_cma + 0.14 * np.sqrt(np.trace(np.cov(elite.T)) / 2), 0.12, 1.8))
    states.append((mean.copy(), sigma_cma, cov.copy(), samples.copy()))


def ellipse_patch(mean, sigma, cov):
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    return Ellipse(mean, 2 * sigma * np.sqrt(vals[0]), 2 * sigma * np.sqrt(vals[1]), angle=angle, fill=False, lw=2.5, color="#16a34a")

fig, ax = plt.subplots(figsize=(8, 7))
draw_himmelblau(ax, "CMA-ES")
pts = ax.scatter([], [], s=20, color="#111827", alpha=0.45, label="кандидаты")
center = ax.scatter([], [], marker="*", s=220, color="#facc15", edgecolor="black", label="центр")
ax.legend(loc="upper right")
ellipses = []


def update(frame):
    for e in ellipses:
        e.remove()
    ellipses.clear()
    mean, sigma_now, cov_now, samples = states[frame]
    pts.set_offsets(samples)
    center.set_offsets([mean])
    ell = ellipse_patch(mean, sigma_now, cov_now)
    ax.add_patch(ell)
    ellipses.append(ell)
    ax.set_title(f"CMA-ES: поколение {frame + 1}")
    return pts, center, ell

anim = animation.FuncAnimation(fig, update, frames=len(states), interval=180, blit=False)
show_video(anim)


## TPE

Tree-structured Parzen Estimator делит наблюдения на хорошие и плохие по квантилю и строит две плотности:

$$
l(x)=p(x\mid f(x)\le y_\gamma),\qquad g(x)=p(x\mid f(x)>y_\gamma).
$$

Следующая точка выбирается там, где отношение $l(x)/g(x)$ велико.

In [ ]:
#@title TPE: анимация good/bad плотностей
N_INIT_TPE = 12 #@param {type:"slider", min:5, max:50, step:1}
N_ITER_TPE = 35 #@param {type:"slider", min:5, max:120, step:5}
GAMMA_TPE = 0.25 #@param {type:"number"}
SEED_TPE = 23 #@param {type:"integer"}

rng = np.random.default_rng(SEED_TPE)
xs = list(rng.uniform(-10, 10, N_INIT_TPE))


def normal_pdf(z):
    return np.exp(-0.5 * z**2) / np.sqrt(2 * np.pi)


def kde_1d(points, grid):
    points = np.asarray(points, dtype=float)
    if len(points) == 0:
        return np.full_like(grid, 1e-12)
    scale = np.std(points) if len(points) > 1 else 1.0
    h = max(0.28, 1.06 * scale * len(points) ** (-1 / 5))
    return normal_pdf((grid[:, None] - points[None, :]) / h).mean(axis=1) / h + 1e-12

states = []
for k in range(N_ITER_TPE):
    ys = J(xs)
    threshold = np.quantile(ys, GAMMA_TPE)
    xs_arr = np.asarray(xs)
    good = xs_arr[ys <= threshold]
    bad = xs_arr[ys > threshold]
    l = kde_1d(good, X)
    g = kde_1d(bad, X)
    ratio = l / g
    pool = rng.uniform(-10, 10, 1200)
    pool_ratio = np.interp(pool, X, ratio)
    top = np.argsort(pool_ratio)[-30:]
    x_new = float(pool[rng.choice(top)])
    states.append((xs_arr.copy(), ys.copy(), good.copy(), bad.copy(), ratio.copy(), x_new))
    xs.append(x_new)

fig, ax = draw_1d("TPE")


def draw_tpe_state(frame):
    ax.clear()
    fig, _ = draw_1d("TPE", size=(13, 5))
    plt.close(fig)


def update(frame):
    ax.clear()
    ax.plot(X, Y, color="#1f2937", lw=2.0, label="целевая функция")
    ax.set_xlim(-10, 10)
    ax.set_ylim(Y_MIN - 0.35, Y_MAX + 0.75)
    ax.set_xlabel("$x$")
    ax.set_ylabel("$f(x)$")
    ax.grid(linestyle=":", alpha=0.65)
    xs_arr, ys, good, bad, ratio, x_new = states[frame]
    ax.scatter(bad, J(bad), marker="x", s=45, color="#ef4444", label="bad/обычные")
    ax.scatter(good, J(good), marker="o", s=55, color="#16a34a", edgecolor="black", label="good")
    ax.axvline(x_new, color="#7c3aed", lw=2.0, label="следующая точка")
    ratio_scaled = (ratio / np.nanmax(ratio)) * 0.9 + (Y_MAX - 0.7)
    ax.plot(X, ratio_scaled, color="#7c3aed", lw=1.8, label="$l(x)/g(x)$")
    ax.set_title(f"TPE: итерация {frame + 1}")
    ax.legend(loc="upper right")
    return []

anim = animation.FuncAnimation(fig, update, frames=len(states), interval=250, blit=False)
show_video(anim)


## MAP-Elites

MAP-Elites хранит не одну лучшую точку, а архив элит по ячейкам пространства поведенческих дескрипторов. Ниже каждая ячейка хранит лучшее найденное значение $f(x)$ для своей ниши.

In [ ]:
#@title MAP-Elites: анимация заполнения архива
GRID_ME = 18 #@param {type:"slider", min:6, max:40, step:1}
N_EVALS_ME = 900 #@param {type:"slider", min:100, max:3000, step:100}
MUTATION_ME = 0.07 #@param {type:"number"}
SEED_ME = 29 #@param {type:"integer"}

rng = np.random.default_rng(SEED_ME)


def quality_from_descriptor(b):
    x = b[..., 0]
    y = b[..., 1]
    return (
        0.7 * (x - 0.78) ** 2
        + 0.9 * (y - 0.24) ** 2
        + 0.18 * np.sin(5 * np.pi * x) ** 2
        + 0.12 * np.cos(4 * np.pi * y) ** 2
        + 0.08 * np.sin(3 * np.pi * (x + y)) ** 2
    )

archive = np.full((GRID_ME, GRID_ME), np.nan)
filled = []
states = []
for t in range(N_EVALS_ME):
    if len(filled) < 20 or rng.random() < 0.35:
        b = rng.random(2)
    else:
        cell = filled[int(rng.integers(len(filled)))]
        center = (np.array(cell) + 0.5) / GRID_ME
        b = np.clip(center + rng.normal(scale=MUTATION_ME, size=2), 0, 1)
    val = float(quality_from_descriptor(b))
    i = min(GRID_ME - 1, int(b[0] * GRID_ME))
    j = min(GRID_ME - 1, int(b[1] * GRID_ME))
    if np.isnan(archive[j, i]) or val < archive[j, i]:
        archive[j, i] = val
        if (i, j) not in filled:
            filled.append((i, j))
    if t % max(1, N_EVALS_ME // 60) == 0 or t == N_EVALS_ME - 1:
        states.append((archive.copy(), t + 1))

fig, ax = plt.subplots(figsize=(8, 7))
cbar_holder = {"cbar": None}


def update(frame):
    ax.clear()
    state, evals = states[frame]
    cmap = plt.cm.viridis_r.copy()
    cmap.set_bad("#f3f4f6")
    im = ax.imshow(state, origin="lower", extent=[0, 1, 0, 1], cmap=cmap, vmin=0, vmax=0.55, aspect="equal")
    ax.set_title(f"MAP-Elites: заполнение архива, {evals} оценок")
    ax.set_xlabel("дескриптор поведения $b_1$")
    ax.set_ylabel("дескриптор поведения $b_2$")
    ax.set_xticks(np.linspace(0, 1, 6))
    ax.set_yticks(np.linspace(0, 1, 6))
    ax.grid(color="white", lw=0.55, alpha=0.9)
    if cbar_holder["cbar"] is None:
        cbar_holder["cbar"] = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar_holder["cbar"].set_label("лучшее найденное $f(x)$, меньше лучше")
    return []

anim = animation.FuncAnimation(fig, update, frames=len(states), interval=180, blit=False)
show_video(anim)


# Что менять в Colab

* `sigma` в первой ячейке: шум oracle.
* `N_*`: бюджет вычислений или число итераций.
* `SEED_*`: разные случайные траектории.
* `STEP_*`, `T0`, `COOLING`, `OMEGA`, `C1`, `C2`: баланс exploration/exploitation.

Все примеры намеренно маленькие, чтобы анимации быстро считались в Colab.